# AquaCrisis supplementary evaluation metrics

This notebook creates ACL-style supplementary tables and classwise F1 plots using only:

- pooled five-fold out-of-fold results for supervised baselines;
- all 1,500 gold-standard posts for LLM evaluation;
- no old 80/20 baseline results;
- no 500-post or 800-post LLM evaluations.



## Imports and shared configuration

In [ ]:
"""
Create ACL-style supplementary evaluation tables and classwise F1 plots
for AquaCrisis.

Inputs
------
- cv_summary(1).csv:
    Five-fold cross-validation summary for supervised baselines.
- pooled_per_class.csv:
    Pooled out-of-fold classwise metrics for supervised baselines.
- llm_eval_summary_all_with_task_b_grouped(3).csv:
    Overall metrics for Gemma 4, GPT-4.1-mini, and Qwen 3.5.
- llm_eval_per_class_all_with_task_b_grouped(3).csv:
    Classwise metrics for Gemma 4, GPT-4.1-mini, and Qwen 3.5.
- batch_overall_metrics(2).csv:
    Overall GPT-4o-mini metrics on the 1,500-post gold set.
- batch_classwise_metrics.csv:
    Classwise GPT-4o-mini metrics on the 1,500-post gold set.

Outputs
-------
- Two overall LaTeX tables with the same row/column structure as Table 3:
    * overall_accuracy.tex
    * overall_weighted_f1.tex
- Compact classwise tables for the best baseline and best LLM per task.
- Full all-system classwise longtables.
- Full CSV files.
- One classwise F1 heatmap per task in PNG and PDF.
- A ready-to-edit supplementary section file.

The script intentionally:
1. Uses only pooled five-fold out-of-fold baseline results.
2. Uses only LLM rows evaluated on all 1,500 gold-standard posts.
3. Does not use the old 80/20 split.
"""

from __future__ import annotations

from pathlib import Path
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 0. Paths and options

In [ ]:
BASE_DIR = Path("/mnt/data")

CV_SUMMARY = BASE_DIR / "cv_summary(1).csv"
BASELINE_CLASSWISE = BASE_DIR / "pooled_per_class.csv"

LLM_OVERALL = BASE_DIR / "llm_eval_summary_all_with_task_b_grouped(3).csv"
LLM_CLASSWISE = BASE_DIR / "llm_eval_per_class_all_with_task_b_grouped(3).csv"

GPT4O_OVERALL = BASE_DIR / "batch_overall_metrics(2).csv"
GPT4O_CLASSWISE = BASE_DIR / "batch_classwise_metrics.csv"

OUT_DIR = BASE_DIR / "aquacrisis_supplementary_metrics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Generate very large all-system classwise LaTeX tables as well as the
# recommended compact selected-system tables and heatmaps.
WRITE_FULL_CLASSWISE_LATEX = False

# Annotating every heatmap cell is useful in a supplement but can be
# disabled if the figure becomes visually dense.
ANNOTATE_HEATMAPS = True

## 1. Display names, task names, class order, and row order

In [ ]:
TASK_NAMES = {
    "task_a": "Task A",
    "task_b": "Task B",
    "task_b_grouped": "Task B.1",
}

BASELINE_MODEL_NAMES = {
    "majority": "Majority",
    "keyword_rules": "Keyword rules",
    "tfidf_logreg": "TF-IDF + LR",
    "multilingual_embeddings": "Multilingual embeddings + LR",
}

LLM_MODEL_NAMES = {
    "gemma4:e4b": "Gemma 4",
    "gpt-4.1-mini": "GPT-4.1-mini",
    "qwen3.5:9b": "Qwen 3.5",
}

MODEL_ORDER = {
    "Majority": 0,
    "Keyword rules": 1,
    "TF-IDF + LR": 2,
    "Multilingual embeddings + LR": 3,
    "Gemma 4": 4,
    "GPT-4.1-mini": 5,
    "GPT-4o-mini": 6,
    "Qwen 3.5": 7,
}

TEXT_ORDER = {
    "all": 0,
    "translated": 1,
    "native": 2,
}

SETTING_ORDER = {
    "Stratified 5-fold CV": 0,
    "rule-based": 0,
    "zero-shot": 1,
    "five-shot": 2,
}

CLASS_ORDER = {
    "task_a": [
        "Water quality / safety / public health",
        "Service disruption / repair / infrastructure",
        "Water conservation / drought / demand management",
        "Flood / stormwater / wastewater / sewer",
        "Environmental sustainability / waste / recycling / biodiversity",
        "Public education / heritage / community engagement",
        "Routine / institutional / customer service / other",
    ],
    "task_b": [
        "Alert / warning",
        "Instruction / advice to public",
        "Operational update / resolution",
        "Reassurance / safety information",
        "Education / awareness",
        "Institutional promotion / community news",
        "Other / unclear",
    ],
    "task_b_grouped": [
        "Risk / incident communication",
        "Education / awareness",
        "Institutional / community communication",
        "Other / unclear",
    ],
}

CLASS_CODES = {
    "task_a": {
        label: f"A{i}"
        for i, label in enumerate(CLASS_ORDER["task_a"], start=1)
    },
    "task_b": {
        label: f"B{i}"
        for i, label in enumerate(CLASS_ORDER["task_b"], start=1)
    },
    "task_b_grouped": {
        label: f"G{i}"
        for i, label in enumerate(CLASS_ORDER["task_b_grouped"], start=1)
    },
}

## 2. Helpers

In [ ]:
def parse_gpt4o_experiment(experiment: str) -> tuple[str, str]:
    """Return (text, setting) from a GPT-4o-mini experiment name."""
    value = str(experiment).lower()
    text = "native" if "native" in value else "translated"
    setting = "zero-shot" if "zero" in value else "five-shot"
    return text, setting


def escape_latex(value: object) -> str:
    """Escape ordinary text for LaTeX table cells."""
    if pd.isna(value):
        return "--"

    text = str(value)
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }

    # Replace backslash first, then the remaining special characters.
    text = text.replace("\\", replacements["\\"])
    for char in ["&", "%", "$", "#", "_", "{", "}", "~", "^"]:
        text = text.replace(char, replacements[char])
    return text


def format_metric(value: object, digits: int = 3) -> str:
    if pd.isna(value):
        return "--"
    return f"{float(value):.{digits}f}"


def format_percent(value: object) -> str:
    if pd.isna(value):
        return "--"
    value = float(value)
    if np.isclose(value, 0.0):
        return "0.0"
    return f"{value:.2f}"


def add_sort_columns(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame["_model_order"] = frame["Model"].map(MODEL_ORDER)
    frame["_text_order"] = frame["Text"].map(TEXT_ORDER)
    frame["_setting_order"] = frame["Setting"].map(SETTING_ORDER)
    return frame.sort_values(
        ["_model_order", "_text_order", "_setting_order"]
    ).drop(columns=["_model_order", "_text_order", "_setting_order"])


def table3_style_latex(
    frame: pd.DataFrame,
    metric_name: str,
    label: str,
    caption: str,
    output_path: Path,
) -> None:
    """
    Write an ACL-style table* with the same structure as the main paper's
    Table 3: Model, Text, Setting, Task A, Task B, Task B.1,
    Invalid Rows %.
    """
    columns = [
        "Model",
        "Text",
        "Setting",
        "Task A",
        "Task B",
        "Task B.1",
        "Invalid Rows %",
    ]
    frame = frame[columns].copy()

    lines = [
        r"\begin{table*}[t]",
        r"\centering",
        r"\small",
        r"\setlength{\tabcolsep}{4pt}",
        r"\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}lllrrrr@{}}",
        r"\toprule",
        rf"Model & Text & Setting & Task A & Task B & Task B.1 & Invalid Rows \% \\",
        r"\midrule",
    ]

    for _, row in frame.iterrows():
        lines.append(
            "{} & {} & {} & {} & {} & {} & {} \\\\".format(
                escape_latex(row["Model"]),
                escape_latex(row["Text"]),
                escape_latex(row["Setting"]),
                format_metric(row["Task A"]),
                format_metric(row["Task B"]),
                format_metric(row["Task B.1"]),
                format_percent(row["Invalid Rows %"]),
            )
        )

    lines.extend(
        [
            r"\bottomrule",
            r"\end{tabular*}",
            rf"\caption{{{caption}}}",
            rf"\label{{{label}}}",
            r"\end{table*}",
            "",
        ]
    )
    output_path.write_text("\n".join(lines), encoding="utf-8")


def classwise_latex(
    frame: pd.DataFrame,
    task_key: str,
    label: str,
    caption: str,
    output_path: Path,
    longtable: bool,
) -> None:
    """
    Write a classwise table. The first three columns mirror the main table,
    followed by class, precision, recall, F1, and support.
    """
    frame = frame.copy()
    frame["Class code"] = frame["Class"].map(CLASS_CODES[task_key])
    frame["_class_order"] = frame["Class"].map(
        {label_: i for i, label_ in enumerate(CLASS_ORDER[task_key])}
    )
    frame["_system_type_order"] = frame["System type"].map(
        {"Baseline": 0, "LLM": 1}
    )
    frame["_model_order"] = frame["Model"].map(MODEL_ORDER)
    frame["_text_order"] = frame["Text"].map(TEXT_ORDER)
    frame["_setting_order"] = frame["Setting"].map(SETTING_ORDER)

    frame = frame.sort_values(
        [
            "_system_type_order",
            "_model_order",
            "_text_order",
            "_setting_order",
            "_class_order",
        ]
    ).drop(
        columns=[
            "_system_type_order",
            "_model_order",
            "_text_order",
            "_setting_order",
            "_class_order",
        ]
    )

    if longtable:
        lines = [
            r"\begingroup",
            r"\small",
            r"\setlength{\tabcolsep}{3.5pt}",
            r"\begin{longtable}{lllcrrrr}",
            rf"\caption{{{caption}}}\label{{{label}}}\\",
            r"\toprule",
            r"Model & Text & Setting & Class & Precision & Recall & F1 & Support \\",
            r"\midrule",
            r"\endfirsthead",
            r"\multicolumn{8}{c}{\tablename\ \thetable\ -- continued} \\",
            r"\toprule",
            r"Model & Text & Setting & Class & Precision & Recall & F1 & Support \\",
            r"\midrule",
            r"\endhead",
            r"\midrule",
            r"\multicolumn{8}{r}{Continued on next page} \\",
            r"\endfoot",
            r"\bottomrule",
            r"\endlastfoot",
        ]
    else:
        lines = [
            r"\begin{table*}[t]",
            r"\centering",
            r"\small",
            r"\setlength{\tabcolsep}{3.5pt}",
            r"\begin{tabular*}{\textwidth}{@{\extracolsep{\fill}}lllcrrrr@{}}",
            r"\toprule",
            r"Model & Text & Setting & Class & Precision & Recall & F1 & Support \\",
            r"\midrule",
        ]

    for _, row in frame.iterrows():
        lines.append(
            "{} & {} & {} & {} & {} & {} & {} & {} \\\\".format(
                escape_latex(row["Model"]),
                escape_latex(row["Text"]),
                escape_latex(row["Setting"]),
                escape_latex(row["Class code"]),
                format_metric(row["Precision"]),
                format_metric(row["Recall"]),
                format_metric(row["F1"]),
                int(row["Support"]),
            )
        )

    if longtable:
        lines.extend([r"\end{longtable}", r"\endgroup", ""])
    else:
        lines.extend(
            [
                r"\bottomrule",
                r"\end{tabular*}",
                rf"\caption{{{caption}}}",
                rf"\label{{{label}}}",
                r"\end{table*}",
                "",
            ]
        )

    output_path.write_text("\n".join(lines), encoding="utf-8")

## 3. Load five-fold supervised baseline overall metrics

In [ ]:
cv = pd.read_csv(CV_SUMMARY)

required_cv = {
    "task",
    "model",
    "variant",
    "folds",
    "pooled_n_eval",
    "pooled_accuracy",
    "pooled_macro_f1",
    "pooled_weighted_f1",
}
missing = required_cv.difference(cv.columns)
if missing:
    raise ValueError(f"Missing columns in CV summary: {sorted(missing)}")

cv = cv[
    (cv["folds"] == 5)
    & (cv["pooled_n_eval"] == 1500)
].copy()

if cv.empty:
    raise ValueError("No five-fold, 1,500-post baseline rows were found.")

baseline_overall = pd.DataFrame(
    {
        "Model": cv["model"].map(BASELINE_MODEL_NAMES),
        "Text": cv["variant"].replace({"original": "native"}),
        "Setting": np.where(
            cv["model"].eq("keyword_rules"),
            "rule-based",
            "Stratified 5-fold CV",
        ),
        "Task key": cv["task"],
        "Task": cv["task"].map(TASK_NAMES),
        "Accuracy": cv["pooled_accuracy"],
        "Macro F1": cv["pooled_macro_f1"],
        "Weighted F1": cv["pooled_weighted_f1"],
        "Invalid Rows %": 0.0,
        "Evaluation source": "Pooled five-fold OOF",
        "System type": "Baseline",
    }
)

if baseline_overall[["Model", "Task"]].isna().any().any():
    raise ValueError("Unmapped baseline model or task name.")

## 4. Load 1,500-post LLM overall metrics

In [ ]:
llm = pd.read_csv(LLM_OVERALL)

required_llm = {
    "eval_set",
    "experiment_id",
    "model",
    "shot",
    "text_source",
    "task",
    "n_gold",
    "missing_predictions",
    "invalid_predictions",
    "accuracy",
    "macro_f1",
    "weighted_f1",
}
missing = required_llm.difference(llm.columns)
if missing:
    raise ValueError(f"Missing columns in LLM overall file: {sorted(missing)}")

llm = llm[
    (llm["eval_set"] == 1500)
    & (llm["n_gold"] == 1500)
].copy()

if llm.empty:
    raise ValueError("No 1,500-post LLM rows were found.")

llm["Model"] = llm["model"].map(LLM_MODEL_NAMES)
llm["Text"] = llm["text_source"].replace({"original": "native"})
llm["Setting"] = llm["shot"].map(
    {"zero": "zero-shot", "five": "five-shot"}
)
llm["Task key"] = llm["task"]
llm["Task"] = llm["task"].map(TASK_NAMES)

# The input file stores invalid/missing counts separately by task.
# To retain the original paper's single experiment-level "Invalid Rows %"
# column, use the maximum task-level invalid-or-missing count for each
# experiment. This is computed ONLY from the 1,500-post evaluation.
llm["_invalid_or_missing"] = llm[
    ["invalid_predictions", "missing_predictions"]
].max(axis=1)

experiment_invalid = (
    llm.groupby("experiment_id", as_index=False)
    .agg(
        invalid_rows=("_invalid_or_missing", "max"),
        n_gold=("n_gold", "max"),
    )
)
experiment_invalid["Invalid Rows %"] = (
    100.0
    * experiment_invalid["invalid_rows"]
    / experiment_invalid["n_gold"]
)
llm = llm.merge(
    experiment_invalid[["experiment_id", "Invalid Rows %"]],
    on="experiment_id",
    how="left",
)

llm_overall = llm[
    [
        "Model",
        "Text",
        "Setting",
        "Task key",
        "Task",
        "accuracy",
        "macro_f1",
        "weighted_f1",
        "Invalid Rows %",
    ]
].rename(
    columns={
        "accuracy": "Accuracy",
        "macro_f1": "Macro F1",
        "weighted_f1": "Weighted F1",
    }
)
llm_overall["Evaluation source"] = "Full 1,500-post gold set"
llm_overall["System type"] = "LLM"

if llm_overall[["Model", "Task", "Setting"]].isna().any().any():
    raise ValueError("Unmapped LLM model, task, or setting name.")

## 5. Load 1,500-post GPT-4o-mini overall metrics

In [ ]:
gpt4o = pd.read_csv(GPT4O_OVERALL)

required_gpt4o = {
    "experiment",
    "task",
    "n_eval",
    "invalid_rate",
    "accuracy_penalized",
    "macro_f1_penalized",
    "weighted_f1_penalized",
}
missing = required_gpt4o.difference(gpt4o.columns)
if missing:
    raise ValueError(f"Missing columns in GPT-4o overall file: {sorted(missing)}")

gpt4o = gpt4o[gpt4o["n_eval"] == 1500].copy()
if gpt4o.empty:
    raise ValueError("No 1,500-post GPT-4o-mini rows were found.")

parsed = gpt4o["experiment"].apply(parse_gpt4o_experiment)
gpt4o[["Text", "Setting"]] = pd.DataFrame(
    parsed.tolist(), index=gpt4o.index
)

gpt4o_overall = pd.DataFrame(
    {
        "Model": "GPT-4o-mini",
        "Text": gpt4o["Text"],
        "Setting": gpt4o["Setting"],
        "Task key": gpt4o["task"],
        "Task": gpt4o["task"].map(TASK_NAMES),
        "Accuracy": gpt4o["accuracy_penalized"],
        "Macro F1": gpt4o["macro_f1_penalized"],
        "Weighted F1": gpt4o["weighted_f1_penalized"],
        "Invalid Rows %": 100.0 * gpt4o["invalid_rate"],
        "Evaluation source": "Full 1,500-post gold set",
        "System type": "LLM",
    }
)

if gpt4o_overall["Task"].isna().any():
    raise ValueError("Unmapped GPT-4o-mini task name.")

## 6. Combine and validate overall metrics

In [ ]:
overall_long = pd.concat(
    [baseline_overall, llm_overall, gpt4o_overall],
    ignore_index=True,
)

duplicates = overall_long.duplicated(
    ["Model", "Text", "Setting", "Task"], keep=False
)
if duplicates.any():
    problem = overall_long.loc[
        duplicates, ["Model", "Text", "Setting", "Task"]
    ]
    raise ValueError(
        "Duplicate overall metric rows found:\n"
        + problem.to_string(index=False)
    )

overall_long = add_sort_columns(overall_long)
overall_long.to_csv(OUT_DIR / "overall_metrics_long.csv", index=False)


def make_overall_wide(metric_column: str) -> pd.DataFrame:
    wide = (
        overall_long.pivot_table(
            index=["Model", "Text", "Setting"],
            columns="Task",
            values=metric_column,
            aggfunc="first",
        )
        .reset_index()
    )

    for task_name in ["Task A", "Task B", "Task B.1"]:
        if task_name not in wide.columns:
            wide[task_name] = np.nan

    invalid = (
        overall_long.groupby(
            ["Model", "Text", "Setting"], as_index=False
        )["Invalid Rows %"]
        .max()
    )
    wide = wide.merge(
        invalid,
        on=["Model", "Text", "Setting"],
        how="left",
    )
    return add_sort_columns(wide)


accuracy_wide = make_overall_wide("Accuracy")
weighted_wide = make_overall_wide("Weighted F1")

accuracy_wide.to_csv(OUT_DIR / "overall_accuracy.csv", index=False)
weighted_wide.to_csv(OUT_DIR / "overall_weighted_f1.csv", index=False)

table3_style_latex(
    accuracy_wide,
    metric_name="Accuracy",
    label="tab:supp-overall-accuracy",
    caption=(
        "Overall accuracy on the 1,500-post gold standard. "
        "Supervised baseline scores are computed from pooled out-of-fold "
        "predictions under stratified five-fold cross-validation; keyword "
        "rules and LLMs are evaluated on all 1,500 annotated posts. "
        "Invalid Rows \\% is computed from the 1,500-post LLM evaluation "
        "and denotes outputs with a missing or invalid task label."
    ),
    output_path=OUT_DIR / "overall_accuracy.tex",
)

table3_style_latex(
    weighted_wide,
    metric_name="Weighted F1",
    label="tab:supp-overall-weighted-f1",
    caption=(
        "Overall weighted-F1 on the 1,500-post gold standard. "
        "Supervised baseline scores are computed from pooled out-of-fold "
        "predictions under stratified five-fold cross-validation; keyword "
        "rules and LLMs are evaluated on all 1,500 annotated posts. "
        "Invalid Rows \\% is computed from the 1,500-post LLM evaluation "
        "and denotes outputs with a missing or invalid task label."
    ),
    output_path=OUT_DIR / "overall_weighted_f1.tex",
)

## 7. Load and combine classwise metrics

In [ ]:
baseline_class = pd.read_csv(BASELINE_CLASSWISE)

required_base_class = {
    "task",
    "model",
    "variant",
    "label",
    "precision",
    "recall",
    "f1",
    "support",
}
missing = required_base_class.difference(baseline_class.columns)
if missing:
    raise ValueError(
        f"Missing columns in baseline classwise file: {sorted(missing)}"
    )

baseline_classwise = pd.DataFrame(
    {
        "Model": baseline_class["model"].map(BASELINE_MODEL_NAMES),
        "Text": baseline_class["variant"].replace({"original": "native"}),
        "Setting": np.where(
            baseline_class["model"].eq("keyword_rules"),
            "rule-based",
            "Stratified 5-fold CV",
        ),
        "Task key": baseline_class["task"],
        "Task": baseline_class["task"].map(TASK_NAMES),
        "Class": baseline_class["label"],
        "Precision": baseline_class["precision"],
        "Recall": baseline_class["recall"],
        "F1": baseline_class["f1"],
        "Support": baseline_class["support"],
        "Evaluation source": "Pooled five-fold OOF",
        "System type": "Baseline",
    }
)

# Verify that pooled_per_class.csv is consistent with the five-fold
# pooled metrics in cv_summary. This prevents an old 80/20 classwise
# file from being used accidentally.
baseline_class_check = (
    baseline_classwise.groupby(
        ["Model", "Text", "Setting", "Task key"], as_index=False
    )
    .apply(
        lambda group: pd.Series(
            {
                "Classwise macro F1": group["F1"].mean(),
                "Classwise weighted F1": np.average(
                    group["F1"], weights=group["Support"]
                ),
            }
        ),
        include_groups=False,
    )
    .reset_index(drop=True)
)

baseline_overall_check = baseline_overall[
    [
        "Model",
        "Text",
        "Setting",
        "Task key",
        "Macro F1",
        "Weighted F1",
    ]
]

baseline_check = baseline_class_check.merge(
    baseline_overall_check,
    on=["Model", "Text", "Setting", "Task key"],
    how="left",
    validate="one_to_one",
)

if not np.allclose(
    baseline_check["Classwise macro F1"],
    baseline_check["Macro F1"],
    atol=1e-8,
):
    raise ValueError(
        "pooled_per_class.csv does not match the five-fold pooled "
        "macro-F1 values in cv_summary."
    )

if not np.allclose(
    baseline_check["Classwise weighted F1"],
    baseline_check["Weighted F1"],
    atol=1e-8,
):
    raise ValueError(
        "pooled_per_class.csv does not match the five-fold pooled "
        "weighted-F1 values in cv_summary."
    )

llm_class = pd.read_csv(LLM_CLASSWISE)

required_llm_class = {
    "eval_set",
    "text_source",
    "shot",
    "model",
    "experiment_id",
    "task",
    "class_label",
    "support",
    "precision",
    "recall",
    "f1",
}
missing = required_llm_class.difference(llm_class.columns)
if missing:
    raise ValueError(
        f"Missing columns in LLM classwise file: {sorted(missing)}"
    )

llm_class = llm_class[llm_class["eval_set"] == 1500].copy()

llm_classwise = pd.DataFrame(
    {
        "Model": llm_class["model"].map(LLM_MODEL_NAMES),
        "Text": llm_class["text_source"].replace({"original": "native"}),
        "Setting": llm_class["shot"].map(
            {"zero": "zero-shot", "five": "five-shot"}
        ),
        "Task key": llm_class["task"],
        "Task": llm_class["task"].map(TASK_NAMES),
        "Class": llm_class["class_label"],
        "Precision": llm_class["precision"],
        "Recall": llm_class["recall"],
        "F1": llm_class["f1"],
        "Support": llm_class["support"],
        "Evaluation source": "Full 1,500-post gold set",
        "System type": "LLM",
    }
)

gpt4o_class = pd.read_csv(GPT4O_CLASSWISE)

required_gpt4o_class = {
    "experiment",
    "task",
    "label",
    "precision",
    "recall",
    "f1",
    "support",
}
missing = required_gpt4o_class.difference(gpt4o_class.columns)
if missing:
    raise ValueError(
        f"Missing columns in GPT-4o classwise file: {sorted(missing)}"
    )

parsed = gpt4o_class["experiment"].apply(parse_gpt4o_experiment)
gpt4o_class[["Text", "Setting"]] = pd.DataFrame(
    parsed.tolist(), index=gpt4o_class.index
)

gpt4o_classwise = pd.DataFrame(
    {
        "Model": "GPT-4o-mini",
        "Text": gpt4o_class["Text"],
        "Setting": gpt4o_class["Setting"],
        "Task key": gpt4o_class["task"],
        "Task": gpt4o_class["task"].map(TASK_NAMES),
        "Class": gpt4o_class["label"],
        "Precision": gpt4o_class["precision"],
        "Recall": gpt4o_class["recall"],
        "F1": gpt4o_class["f1"],
        "Support": gpt4o_class["support"],
        "Evaluation source": "Full 1,500-post gold set",
        "System type": "LLM",
    }
)

classwise = pd.concat(
    [baseline_classwise, llm_classwise, gpt4o_classwise],
    ignore_index=True,
)

if classwise[
    ["Model", "Task", "Class", "Precision", "Recall", "F1", "Support"]
].isna().any().any():
    bad = classwise[
        classwise[
            ["Model", "Task", "Class", "Precision", "Recall", "F1", "Support"]
        ].isna().any(axis=1)
    ]
    raise ValueError(
        "Missing or unmapped classwise values:\n"
        + bad.head(20).to_string(index=False)
    )

# Every model/task evaluation should represent the full gold set.
support_check = (
    classwise.groupby(
        ["Model", "Text", "Setting", "Task key"], as_index=False
    )["Support"]
    .sum()
)
if not support_check["Support"].eq(1500).all():
    bad = support_check[~support_check["Support"].eq(1500)]
    raise ValueError(
        "Some classwise groups do not sum to 1,500 gold examples:\n"
        + bad.to_string(index=False)
    )

# Add stable class codes and order.
classwise["Class code"] = classwise.apply(
    lambda row: CLASS_CODES[row["Task key"]][row["Class"]],
    axis=1,
)
classwise["_class_order"] = classwise.apply(
    lambda row: CLASS_ORDER[row["Task key"]].index(row["Class"]),
    axis=1,
)
classwise["_model_order"] = classwise["Model"].map(MODEL_ORDER)
classwise["_text_order"] = classwise["Text"].map(TEXT_ORDER)
classwise["_setting_order"] = classwise["Setting"].map(SETTING_ORDER)

classwise = classwise.sort_values(
    [
        "Task key",
        "_model_order",
        "_text_order",
        "_setting_order",
        "_class_order",
    ]
).drop(
    columns=[
        "_model_order",
        "_text_order",
        "_setting_order",
        "_class_order",
    ]
)

classwise.to_csv(OUT_DIR / "classwise_metrics_all_models.csv", index=False)

## 8. Select strongest baseline and strongest LLM per task

In [ ]:
best_system_rows = []

for task_key, task_name in TASK_NAMES.items():
    task_overall = overall_long[overall_long["Task key"] == task_key]

    for system_type in ["Baseline", "LLM"]:
        subset = task_overall[
            task_overall["System type"] == system_type
        ].copy()

        best = subset.sort_values(
            ["Macro F1", "Accuracy"],
            ascending=[False, False],
        ).iloc[0]

        best_system_rows.append(
            {
                "Task key": task_key,
                "Task": task_name,
                "System type": system_type,
                "Model": best["Model"],
                "Text": best["Text"],
                "Setting": best["Setting"],
                "Macro F1": best["Macro F1"],
            }
        )

best_systems = pd.DataFrame(best_system_rows)
best_systems.to_csv(OUT_DIR / "selected_best_systems.csv", index=False)

selected_parts = []
for _, best in best_systems.iterrows():
    mask = (
        classwise["Task key"].eq(best["Task key"])
        & classwise["Model"].eq(best["Model"])
        & classwise["Text"].eq(best["Text"])
        & classwise["Setting"].eq(best["Setting"])
    )
    selected_parts.append(classwise[mask].copy())

selected_classwise = pd.concat(selected_parts, ignore_index=True)
selected_classwise.to_csv(
    OUT_DIR / "classwise_metrics_selected_systems.csv",
    index=False,
)

## 9. Write compact and full classwise LaTeX tables

In [ ]:
for task_key, task_name in TASK_NAMES.items():
    slug = {
        "task_a": "task_a",
        "task_b": "task_b",
        "task_b_grouped": "task_b1",
    }[task_key]

    selected_task = selected_classwise[
        selected_classwise["Task key"] == task_key
    ].copy()

    classwise_latex(
        selected_task,
        task_key=task_key,
        label=f"tab:supp-classwise-{slug}",
        caption=(
            f"Classwise precision, recall, and F1 for the strongest "
            f"supervised baseline and strongest LLM on {task_name}. "
            "Baseline values use pooled out-of-fold predictions from "
            "stratified five-fold cross-validation; LLM values use all "
            "1,500 gold-standard posts. Support is the number of gold "
            "examples in each class. Class codes are defined in "
            "Table~\\ref{tab:supp-class-label-key}."
        ),
        output_path=OUT_DIR / f"classwise_selected_{slug}.tex",
        longtable=False,
    )

    if WRITE_FULL_CLASSWISE_LATEX:
        all_task = classwise[
            classwise["Task key"] == task_key
        ].copy()

        classwise_latex(
            all_task,
            task_key=task_key,
            label=f"tab:supp-classwise-all-{slug}",
            caption=(
                f"Complete classwise precision, recall, and F1 for all "
                f"systems on {task_name}. Baseline values use pooled "
                "five-fold out-of-fold predictions; LLM values use all "
                "1,500 gold-standard posts. Support is the number of gold "
                "examples. Class codes are defined in "
                "Table~\\ref{tab:supp-class-label-key}."
            ),
            output_path=OUT_DIR / f"classwise_all_{slug}.tex",
            longtable=True,
        )

## 10. Label-key table

In [ ]:
label_key_lines = [
    r"\begin{table*}[t]",
    r"\centering",
    r"\small",
    r"\begin{tabular}{lll}",
    r"\toprule",
    r"Task & Code & Class label \\",
    r"\midrule",
]

for task_key, task_name in TASK_NAMES.items():
    for class_label in CLASS_ORDER[task_key]:
        label_key_lines.append(
            "{} & {} & {} \\\\".format(
                escape_latex(task_name),
                escape_latex(CLASS_CODES[task_key][class_label]),
                escape_latex(class_label),
            )
        )

label_key_lines.extend(
    [
        r"\bottomrule",
        r"\end{tabular}",
        r"\caption{Class codes used in the supplementary classwise tables and figures.}",
        r"\label{tab:supp-class-label-key}",
        r"\end{table*}",
        "",
    ]
)

(OUT_DIR / "class_label_key.tex").write_text(
    "\n".join(label_key_lines),
    encoding="utf-8",
)

## 11. Create one all-system classwise F1 heatmap per task

In [ ]:
def short_row_label(row: pd.Series) -> str:
    model = row["Model"].replace(
        "Multilingual embeddings + LR", "mEmb + LR"
    )
    setting = {
        "Stratified 5-fold CV": "5-fold",
        "rule-based": "rules",
        "zero-shot": "0s",
        "five-shot": "5s",
    }[row["Setting"]]

    if row["Text"] == "all":
        return f"{model} | {setting}"
    return f"{model} | {row['Text']} | {setting}"


for task_key, task_name in TASK_NAMES.items():
    task_data = classwise[classwise["Task key"] == task_key].copy()

    task_data["System"] = task_data.apply(short_row_label, axis=1)

    system_order = (
        task_data[
            ["System", "Model", "Text", "Setting"]
        ]
        .drop_duplicates()
    )
    system_order["_model_order"] = system_order["Model"].map(MODEL_ORDER)
    system_order["_text_order"] = system_order["Text"].map(TEXT_ORDER)
    system_order["_setting_order"] = system_order["Setting"].map(
        SETTING_ORDER
    )
    system_order = system_order.sort_values(
        ["_model_order", "_text_order", "_setting_order"]
    )["System"].tolist()

    class_codes = [
        CLASS_CODES[task_key][label]
        for label in CLASS_ORDER[task_key]
    ]

    heat = task_data.pivot_table(
        index="System",
        columns="Class code",
        values="F1",
        aggfunc="first",
    )
    heat = heat.reindex(index=system_order, columns=class_codes)

    fig_height = max(6.5, 0.38 * len(system_order) + 1.8)
    fig, ax = plt.subplots(figsize=(10.5, fig_height))

    image = ax.imshow(
        heat.to_numpy(dtype=float),
        aspect="auto",
        vmin=0.0,
        vmax=1.0,
    )

    ax.set_xticks(np.arange(len(class_codes)))
    ax.set_xticklabels(class_codes, fontsize=10, fontweight="bold")
    ax.set_yticks(np.arange(len(system_order)))
    ax.set_yticklabels(system_order, fontsize=8.5)

    ax.set_xlabel("Class", fontsize=11, fontweight="bold")
    ax.set_ylabel("Model, text, and setting", fontsize=11, fontweight="bold")
    ax.set_title(
        f"Classwise F1 on {task_name}",
        fontsize=13,
        fontweight="bold",
    )

    if ANNOTATE_HEATMAPS:
        values = heat.to_numpy(dtype=float)
        threshold = 0.5
        for i in range(values.shape[0]):
            for j in range(values.shape[1]):
                value = values[i, j]
                if np.isnan(value):
                    continue
                ax.text(
                    j,
                    i,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                    fontsize=7,
                )

    colorbar = fig.colorbar(image, ax=ax)
    colorbar.set_label("F1", fontsize=10, fontweight="bold")

    fig.tight_layout()

    slug = {
        "task_a": "task_a",
        "task_b": "task_b",
        "task_b_grouped": "task_b1",
    }[task_key]

    fig.savefig(
        OUT_DIR / f"classwise_f1_heatmap_{slug}.png",
        dpi=300,
        bbox_inches="tight",
    )
    fig.savefig(
        OUT_DIR / f"classwise_f1_heatmap_{slug}.pdf",
        bbox_inches="tight",
    )
    plt.close(fig)

## 12. Ready-to-edit supplementary section and main-text references

In [ ]:
supplement_text = r"""
% Required packages in the preamble:
% \usepackage{booktabs}
% \usepackage{graphicx}
% \usepackage{longtable}   % only needed for the full all-system tables

\section{Additional Evaluation Metrics}
\label{sec:supp-evaluation}

The main paper reports macro-F1 because the gold standard is strongly
imbalanced. For completeness, Tables~\ref{tab:supp-overall-accuracy}
and~\ref{tab:supp-overall-weighted-f1} report accuracy and weighted-F1.
Supervised baseline values are computed from pooled out-of-fold
predictions under stratified five-fold cross-validation. Keyword rules
and LLMs are evaluated on all 1,500 gold-standard posts; no 500-post,
800-post, or 80/20-split result is included.

\input{overall_accuracy.tex}
\input{overall_weighted_f1.tex}

Table~\ref{tab:supp-class-label-key} defines the abbreviated class codes.
Tables~\ref{tab:supp-classwise-task_a}--\ref{tab:supp-classwise-task_b1}
report classwise precision, recall, and F1 for the strongest supervised
baseline and strongest LLM on each task. Complete all-system classwise
values are provided in the accompanying CSV file, and
Figures~\ref{fig:supp-classwise-task_a}--\ref{fig:supp-classwise-task_b1}
visualize classwise F1 for all systems.

\input{class_label_key.tex}
\input{classwise_selected_task_a.tex}
\input{classwise_selected_task_b.tex}
\input{classwise_selected_task_b1.tex}

\begin{figure*}[t]
    \centering
    \includegraphics[width=\textwidth]{classwise_f1_heatmap_task_a.pdf}
    \caption{Classwise F1 for all systems on Task A. Supervised baseline
    values use pooled five-fold out-of-fold predictions, and LLM values
    use all 1,500 gold-standard posts. Class codes are defined in
    Table~\ref{tab:supp-class-label-key}.}
    \label{fig:supp-classwise-task_a}
\end{figure*}

\begin{figure*}[t]
    \centering
    \includegraphics[width=\textwidth]{classwise_f1_heatmap_task_b.pdf}
    \caption{Classwise F1 for all systems on Task B. Supervised baseline
    values use pooled five-fold out-of-fold predictions, and LLM values
    use all 1,500 gold-standard posts. Class codes are defined in
    Table~\ref{tab:supp-class-label-key}.}
    \label{fig:supp-classwise-task_b}
\end{figure*}

\begin{figure*}[t]
    \centering
    \includegraphics[width=\textwidth]{classwise_f1_heatmap_task_b1.pdf}
    \caption{Classwise F1 for all systems on grouped Task B.1. Supervised
    baseline values use pooled five-fold out-of-fold predictions, and
    LLM values use all 1,500 gold-standard posts. Class codes are defined
    in Table~\ref{tab:supp-class-label-key}.}
    \label{fig:supp-classwise-task_b1}
\end{figure*}
""".strip()

(OUT_DIR / "supplementary_section.tex").write_text(
    supplement_text + "\n",
    encoding="utf-8",
)

main_text_reference = r"""
% Suggested sentence at the end of Section 3.6 (Evaluation protocol):
Additional overall and classwise results are reported in
Appendix~\ref{sec:supp-evaluation}. Tables~\ref{tab:supp-overall-accuracy}
and~\ref{tab:supp-overall-weighted-f1} provide accuracy and weighted-F1,
while the classwise tables and figures report precision, recall, and F1.
All supervised results use pooled out-of-fold predictions from
stratified five-fold cross-validation, and all LLM results use the full
1,500-post gold standard.

% Shorter alternative for the Results section:
Additional accuracy, weighted-F1, and classwise results are reported in
Appendix~\ref{sec:supp-evaluation}.
""".strip()

(OUT_DIR / "main_text_reference.tex").write_text(
    main_text_reference + "\n",
    encoding="utf-8",
)

## 13. Write a short README

In [ ]:
readme = """
Recommended ACL supplement layout
=================================

Use:
- overall_accuracy.tex
- overall_weighted_f1.tex
- class_label_key.tex
- classwise_selected_task_a.tex
- classwise_selected_task_b.tex
- classwise_selected_task_b1.tex
- the three classwise_f1_heatmap_*.pdf figures

The compact classwise tables compare the strongest supervised baseline
and strongest LLM for each task. This is recommended because an
all-system classwise table contains 396 rows and is difficult to read
in the ACL two-column layout.

Complete classwise values for every model are retained in:
- classwise_metrics_all_models.csv

To generate full all-system LaTeX longtables, set
WRITE_FULL_CLASSWISE_LATEX = True. Longtable is not ideal in the
standard ACL two-column layout and may require a separate one-column
supplement.

Important filtering
===================
- Baselines: only pooled five-fold out-of-fold results with 1,500
  evaluated posts are used.
- LLMs: only eval_set == 1500 or n_eval == 1500 rows are used.
- The old 80/20 split and the 500/800-post LLM evaluations are excluded.
""".strip()

(OUT_DIR / "README.txt").write_text(readme + "\n", encoding="utf-8")

## 14. Print a concise build report

In [ ]:
print(f"Created supplementary outputs in: {OUT_DIR}")
print("\nSelected systems for compact classwise tables:")
print(
    best_systems[
        ["Task", "System type", "Model", "Text", "Setting", "Macro F1"]
    ].to_string(index=False)
)
print("\nFiles:")
for path in sorted(OUT_DIR.iterdir()):
    print(f"  - {path.name}")

## Outputs

Running all cells creates `/mnt/data/aquacrisis_supplementary_metrics/` and a ZIP bundle containing the LaTeX tables, CSV files, and classwise plots.